In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

# 可从 mock_dataset 或其上级项目目录启动 notebook。
CSV_NAME = "Actual_generation_202001010600_202601010600_Hour.csv"
CANDIDATES = [Path("smard_data") / CSV_NAME,
              Path("mock_dataset/smard_data") / CSV_NAME]
DATA_PATH = next((p for p in CANDIDATES if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("未找到 CSV，请把 DATA_PATH 改为文件的实际路径。")
SOURCE_TIMEZONE = "Europe/Berlin"

plt.rcParams.update({
    "figure.figsize": (12, 5), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
})
# 图内用英文，避免本机缺少中文字体导致乱码；说明文字使用中文。


In [5]:
raw_df = pd.read_csv(DATA_PATH, sep=";", thousands=",", na_values=["-"])
name_map = {c: c.split(" [MWh]")[0] for c in raw_df.columns[2:]}
labels = pd.to_datetime(raw_df["Start date"], format="%b %d, %Y %I:%M %p")
# 原始顺序用于推断秋季两个 02:00；无法推断时抛错，不静默删除。
local_time = labels.dt.tz_localize(
    SOURCE_TIMEZONE, ambiguous="infer", nonexistent="raise"
)
df = raw_df[list(name_map)].rename(columns=name_map).copy()
df = df.apply(pd.to_numeric, errors="raise")
df.index = pd.DatetimeIndex(local_time, name="timestamp")
df = df.sort_index()
utc_steps = df.index.tz_convert("UTC").to_series().diff().dropna()

print(f"Rows: {len(df):,}; generation variables: {df.shape[1]}")
print(f"First hour: {df.index.min()}; last hour starts: {df.index.max()}")
print(f"Duplicated source clock labels: {labels.duplicated().sum()}")
print(f"重复列名: {df.columns.duplicated().sum()}; "
      f"重复时间戳: {df.index.duplicated().sum()}; "
      f"非一小时间隔: {utc_steps.ne(pd.Timedelta(hours=1)).sum()}")
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str), "missing": df.isna().sum(),
    "missing_pct": 100 * df.isna().mean(), "zero_count": df.eq(0).sum(),
    "min_MWh": df.min(), "max_MWh": df.max(),
})
display(summary.round(2))
display(df.head())

# 为当地自然日构造完整的应有小时数，首尾部分日期不会被当完整日。
day_edges = pd.date_range(df.index.min().normalize(),
                          df.index.max().normalize() + pd.DateOffset(days=1),
                          freq="D")
expected_hours = pd.Series(
    np.diff(day_edges.tz_convert("UTC").asi8) / 3.6e12,
    index=day_edges[:-1], name="expected_hours"
)
daily_count = df.resample("D").count()
complete_day = daily_count.eq(expected_hours, axis=0)
daily_energy = df.resample("D").sum(min_count=1).where(complete_day)
daily_mean = df.resample("D").mean().where(complete_day)
print("Expected hours per local day:", expected_hours.value_counts().to_dict())


Rows: 52,608; generation variables: 12
First hour: 2020-01-01 06:00:00+01:00; last hour starts: 2026-01-01 05:00:00+01:00
Duplicated source clock labels: 6
重复列名: 0; 重复时间戳: 0; 非一小时间隔: 0


,dtype,missing,missing_pct,zero_count,min_MWh,max_MWh
Biomass,float64,0,0.00,0,3085.63,5271.25
Hydropower,float64,0,0.00,0,841.00,3082.00
Wind offshore,float64,0,0.00,35,0.00,8278.07
Wind onshore,float64,0,0.00,0,46.50,48499.50
Photovoltaics,float64,0,0.00,4221,0.00,52132.25
Other renewable,float64,0,0.00,0,56.46,244.25
Nuclear,float64,16842,32.01,6949,0.00,8204.75
Lignite,float64,0,0.00,0,1877.75,17173.25
Hard coal,float64,0,0.00,0,135.50,16393.50
Fossil gas,float64,0,0.00,0,1263.50,20146.50


,Biomass,Hydropower,Wind offshore,Wind onshore,Photovoltaics,Other renewable,Nuclear,Lignite,Hard coal,Fossil gas,Hydro pumped storage,Other conventional
timestamp,,,,,,,,,,,,
2020-01-01 06:00:00+01:00,4797.50,1333.25,1943.00,5565.50,0.00,222.75,8076.25,9338.25,2007.00,5427.75,1406.25,1736.00
2020-01-01 07:00:00+01:00,4842.25,1329.25,2154.50,5250.50,0.25,223.25,8093.00,9323.25,2012.50,5609.25,491.75,1740.75
2020-01-01 08:00:00+01:00,4886.75,1318.50,2283.25,5048.50,366.75,223.25,8097.25,9801.50,2050.00,5707.75,202.25,1743.75
2020-01-01 09:00:00+01:00,4890.50,1336.00,2341.25,5038.75,3028.50,222.00,8098.00,9961.75,2066.00,5779.25,180.75,1729.50
2020-01-01 10:00:00+01:00,4852.25,1369.00,2546.50,4775.75,6612.50,221.75,8050.75,10179.00,2045.25,5781.25,453.00,1736.25


Expected hours per local day: {24.0: 2181, 23.0: 6, 25.0: 6}


In [6]:
# 不处理时区：直接计算本地钟表时间的差值
naive_diff = labels.diff()

# 处理时区后：转成 UTC，再计算实际经过的时间
utc_diff = local_time.dt.tz_convert("UTC").diff()

print("未处理时区时的间隔统计：")
print(naive_diff.value_counts(dropna=False).sort_index())

print("\n转成 UTC 后的间隔统计：")
print(utc_diff.value_counts(dropna=False).sort_index())

# 查看本地时间差不是 1 小时的行，并对照 UTC 差值
comparison = pd.DataFrame({
    "previous_local_time": labels.shift(1),
    "current_local_time": labels,
    "naive_diff": naive_diff,
    "utc_diff": utc_diff,
})

anomalies = comparison[
    comparison["naive_diff"].notna()
    & comparison["naive_diff"].ne(pd.Timedelta(hours=1))
]

display(anomalies)

未处理时区时的间隔统计：
Start date
0 days 00:00:00        6
0 days 01:00:00    52595
0 days 02:00:00        6
NaT                    1
Name: count, dtype: int64

转成 UTC 后的间隔统计：
Start date
0 days 01:00:00    52607
NaT                    1
Name: count, dtype: int64


,previous_local_time,current_local_time,naive_diff,utc_diff
2108,2020-03-29 01:00:00,2020-03-29 03:00:00,0 days 02:00:00,0 days 01:00:00
7148,2020-10-25 02:00:00,2020-10-25 02:00:00,0 days 00:00:00,0 days 01:00:00
10844,2021-03-28 01:00:00,2021-03-28 03:00:00,0 days 02:00:00,0 days 01:00:00
16052,2021-10-31 02:00:00,2021-10-31 02:00:00,0 days 00:00:00,0 days 01:00:00
19580,2022-03-27 01:00:00,2022-03-27 03:00:00,0 days 02:00:00,0 days 01:00:00
24788,2022-10-30 02:00:00,2022-10-30 02:00:00,0 days 00:00:00,0 days 01:00:00
28316,2023-03-26 01:00:00,2023-03-26 03:00:00,0 days 02:00:00,0 days 01:00:00
33524,2023-10-29 02:00:00,2023-10-29 02:00:00,0 days 00:00:00,0 days 01:00:00
37220,2024-03-31 01:00:00,2024-03-31 03:00:00,0 days 02:00:00,0 days 01:00:00
42260,2024-10-27 02:00:00,2024-10-27 02:00:00,0 days 00:00:00,0 days 01:00:00
